In [2]:
import datasets
from datasets import load_dataset
import random
import pandas as pd
import re

from string import Template

from vllm import LLM, SamplingParams
import gc
import torch
import time

In [3]:
# Фиксируем seed
SEED = 42
random.seed(SEED)

# Загрузка данных

In [4]:
# Загружаем датасет
dataset = load_dataset("UniqueData/asos-e-commerce-dataset")

# Выбираем split (обычно 'train')
data = dataset['train']  # или другой доступный split

# Выбираем 100 случайных примеров
random_indices = random.sample(range(len(data)), 100)
random_samples = data.select(random_indices)

target_features = ["name", "size", "category", "price", "color"]
result_data = []
for sample in random_samples:
  target_features_sample = {feature_name: sample.get(feature_name) for feature_name in target_features}
  result_data.append(target_features_sample)

df = pd.DataFrame(result_data) 
df.head(5)

,name,size,category,price,color
0,ASOS DESIGN Curve wrap bodysuit with angel sle...,"UK 16,UK 18,UK 20,UK 22,UK 24,UK 26,UK 28,UK 30",ASOS DESIGN Curve wrap bodysuit with angel sle...,25.00,Black
1,Bershka corset detail roll neck jumper in black,"XS - UK 6,S - UK 8,M - UK 10,L - UK 12,XL - UK...",Bershka corset detail roll neck jumper in black,Now 18.50,BLACK
2,Stradivarius oversized faux leather padded puf...,"XS - UK 6,S - UK 8,M - UK 10,L - UK 12,XL - UK 14",Stradivarius oversized faux leather padded puf...,59.99,ECRU
3,New Balance Running 1/2 zip long sleeve top in...,"XS - UK 4-6,S - UK 8-10,M - UK 12-14,L - UK 16...",New Balance Running 1/2 zip long sleeve top in...,40.00,PURPLE
4,ASYOU satin square neck cami dress with diaman...,"UK 4,UK 6,UK 8,UK 10,UK 12,UK 14,UK 16,UK 18",ASYOU satin square neck cami dress with diaman...,42.99,Black


# Загрузка модели

In [5]:
# Удаление модели и освобождение памяти
# del llm
gc.collect()

# Дополнительная очистка GPU памяти (если используется CUDA)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

In [ ]:
model_name = "Qwen/Qwen3-30B-A3B-Instruct-2507"

# Инициализация модели
print("Загрузка модели...")
llm = LLM(
    model=model_name,
    tensor_parallel_size=1,
    max_model_len=20_000
)

In [6]:
# Подготовка параметров генерации
sampling_params = SamplingParams(temperature=0.7,
                                top_p=0.9,
                                max_tokens=200)

# 1. MVP. Простая генерация

## Подготовка данных

In [7]:
prompt_template = Template(
"""
Ты — профессиональный копирайтер маркетплейса. 
Твоя задача — сгенерировать продающее описание товара на основе характеристик, предоставленных продавцом.

Название товара: ${title}
Характеристики:
${characteristics}

СТРОГИЕ ТРЕБОВАНИЯ К ВЫВОДУ:
1. Описание должно начинаться ровно с заголовка: <h1>${title}</h1>
2. Под каждую характеристику создай отдельный абзац. Количество тегов <p>...</p> должно точно совпадать с количеством переданных характеристик.
3. Каждый абзац обязательно обрамляй тегами <p> и </p>. Внутри абзаца раскрой смысл характеристики, объясни практическую пользу для покупателя.
4. В описании должны быть использованы ВСЕ характеристики из списка. Ни одну не пропускай и не объединяй несколько в один абзац.
5. Максимизируй лексическое разнообразие (метрика VocD): активно используй синонимы, разные части речи, варьируй длину и синтаксическую структуру предложений. Избегай повторов однокоренных слов, шаблонных фраз ("высокое качество", "идеальный выбор") и канцеляризмов.
6. Стиль — естественный, информативный, адаптированный под онлайн-покупателя.
7. Выведи ТОЛЬКО готовый HTML-код. Не добавляй вступлений, пояснений, тегов <html>/<body>, markdown-обёрток (```) или подписей.

Сгенерируй описание:
"""
)

In [8]:
def prepare_characteristics(row):
    labels = {
        'size': 'Размеры',
        'category': 'Категория',
        'price': 'Цена',
        'color': 'Цвет'
    }
    chars = []
    for col, label in labels.items():
        val = row.get(col)
        if pd.isna(val) or str(val).strip() in ('', 'None', 'nan'):
            continue
        # Очистка цены от мусора (Now 26.00 -> 26.00)
        if col == 'price':
            val = re.sub(r'^(Now|Was|Price|£|\$|€)\s*', '', str(val), flags=re.IGNORECASE).strip()
        chars.append(f"{label}: {str(val).strip()}")
    return '\n'.join(chars), len(chars)

In [9]:
input_data = []

for _, row in df.iterrows():
    title = str(row['name']).strip()
    if not title or title in ('None', 'nan', ''):
        title = "Без названия"
        
    chars_str, p_count = prepare_characteristics(row)
    
    # Заполняем шаблон
    prompt = prompt_template.substitute(title=title, characteristics=chars_str)
    
    input_data.append({
        'title': title,
        'characteristics_raw': chars_str,
        'expected_p_count': p_count,
        'prompt': prompt
    })

# Проверка: вывод первого промпта
print(input_data[0]['prompt'])
print(f"\nОжидается абзацев <p>: {input_data[0]['expected_p_count']}")


Ты — профессиональный копирайтер маркетплейса. 
Твоя задача — сгенерировать продающее описание товара на основе характеристик, предоставленных продавцом.

Название товара: ASOS DESIGN Curve wrap bodysuit with angel sleeve in black
Характеристики:
Размеры: UK 16,UK 18,UK 20,UK 22,UK 24,UK 26,UK 28,UK 30
Категория: ASOS DESIGN Curve wrap bodysuit with angel sleeve in black
Цена: 25.00
Цвет: Black

СТРОГИЕ ТРЕБОВАНИЯ К ВЫВОДУ:
1. Описание должно начинаться ровно с заголовка: <h1>ASOS DESIGN Curve wrap bodysuit with angel sleeve in black</h1>
2. Под каждую характеристику создай отдельный абзац. Количество тегов <p>...</p> должно точно совпадать с количеством переданных характеристик.
3. Каждый абзац обязательно обрамляй тегами <p> и </p>. Внутри абзаца раскрой смысл характеристики, объясни практическую пользу для покупателя.
4. В описании должны быть использованы ВСЕ характеристики из списка. Ни одну не пропускай и не объединяй несколько в один абзац.
5. Максимизируй лексическое разнообра

## Инференс

In [10]:
from src.inference import SampleGenerationResult, FullGenerationResult, print_compact_stats

def run_generation(input_data, sampling_params, llm):
    start_time = time.time()
    generated_data = llm.generate(input_data, sampling_params)
    end_time = time.time()

    total_time = end_time - start_time
    latency = total_time
    throughput = len(input_data) / total_time

    # В этом случае все ответы мы получаем одновременно,
    # Поэтому можно сказать, что время каждого ответа равно времени полной генерации всех ответов
    # Это основной недостаток офлайн-генерации
    latency_list = [latency for _ in range(len(generated_data))]

    return generated_data, throughput, latency_list


def prepare_results(generated_data, throughput, latency_list):
    generation_results = []
    for output, features, latency in zip(generated_data, result_data, latency_list):
        description = output.outputs[0].text
        score_info = calculate_total_score(output.outputs[0].text, features)

        sample_info = SampleGenerationResult(
            latency=latency,
            description=description,
            total_score=score_info['total_score'],
            detailed_score=copy.deepcopy(score_info['detailed_scores'])
        )
        generation_results.append(sample_info)


    full_generation_result = FullGenerationResult(
        throughput=throughput,
        latency_avg=sum([x.latency for x in generation_results])/len(generation_results),
        total_score_avg=sum([x.total_score for x in generation_results])/len(generation_results),
        outputs=generation_results
    )
    return full_generation_result

In [32]:
generated_data, throughput, latency_list = run_generation(input_data, sampling_params, llm)
print(f"Длительность ответа на единичный запрос {sum(latency_list)/len(latency_list)}")
print(f"Количество запросов, обрабатываемых в секунду {throughput}")

NameError: name 'llm' is not defined

In [ ]:
full_generation_result = prepare_results(generated_data, throughput, latency_list)
print_compact_stats(full_generation_result)

# 2. Повышение эффективности генерации

## Добавление спекулятивного декодинга

In [ ]:
# Инициализация модели
print("Загрузка модели...")
llm = LLM(
    model="Qwen/Qwen3-30B-A3B-Instruct-2507",
    tensor_parallel_size=1,
    max_model_len=20_000,
    speculative_config={
        "method": "ngram",
        "num_speculative_tokens": 5,
        "prompt_lookup_max": 4,
    },
)

generated_data, throughput, latency_list = run_generation(input_data, sampling_params, llm)
print(f"Длительность ответа на единичный запрос {sum(latency_list)/len(latency_list)}")
print(f"Количество запросов, обрабатываемых в секунду {throughput}")

full_generation_result = prepare_results(generated_data, throughput, latency_list)
print_compact_stats(full_generation_result)

## Добавление квантизации

In [ ]:
## для инференса с квантизацией
model_name = "cpatonn/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit"

# Инициализация модели
print("Загрузка модели...")
llm = LLM(
    model=model_name,
    tensor_parallel_size=1,
    max_model_len=20_000
)

generated_data, throughput, latency_list = run_generation(input_data, sampling_params, llm)
print(f"Длительность ответа на единичный запрос {sum(latency_list)/len(latency_list)}")
print(f"Количество запросов, обрабатываемых в секунду {throughput}")

full_generation_result = prepare_results(generated_data, throughput, latency_list)
print_compact_stats(full_generation_result)

# 3. Онлайн-инференс

In [ ]:
!vllm serve "Qwen/Qwen3-30B-A3B-Instruct-2507" \
  --dtype auto \
  --api-key token-abc123 \
  --max-model-len '20k'

In [ ]:
import aiohttp
import asyncio
import time
import nest_asyncio

nest_asyncio.apply()

async def send_to_generation(prompt, model_name="Qwen/Qwen3-30B-A3B-Instruct-2507", sampling_params=None):
    base_url = "http://localhost:8000/v1"
    params = copy.deepcopy(sampling_params) or {}
    params["model_name"] = model_name
    params["prompt"] = prompt
    async with aiohttp.ClientSession() as session:
        start_time = time.time()
        print(f"start {time.strftime('%H:%M:%S')} - {prompt[:20]}...")
        async with session.post(
            f"{base_url}/completions",
            json={
                "model": model_name,
                "prompt": prompt,
                "max_tokens": 500,
                "temperature": 0.7
            },
            headers={"Authorization": "Bearer token-abc123"},
            timeout=100
        ) as response:
            result = await response.json()
            end_time = time.time()
            print(f"end {time.strftime('%H:%M:%S')} - {prompt[:20]}... (duration: {end_time-start_time:.5f}s)")
            return result, start_time, end_time

async def schedule_requests(prompts, pause_duration=1, sampling_params=None, model_name="Qwen/Qwen3-30B-A3B-Instruct-2507"):
    tasks = []

    for i, prompt in enumerate(prompts):
        # Создаём задачу, но не ждём её завершения
        task = asyncio.create_task(send_to_generation(prompt, model_name, sampling_params))
        tasks.append(task)

        # Ждём pause_duration секунд перед следующим запросом
        if i < len(prompts) - 1:  # Не ждём после последнего запроса
            await asyncio.sleep(pause_duration)

    # Ждём завершения всех задач
    results = await asyncio.gather(*tasks)
    return results

# Запуск
prompts = [
    "Качественный сервис это",
    "Объясни понятие 'машинное обучение' простыми словами:",
    "Напиши рецепт быстрого ужина из курицы:",
    "Какие преимущества у использования Python для data science?"
    "Напиши определение объектно ориентированного программирования",
]
results = await schedule_requests(prompts, 2)

In [ ]:
def prepare_results_online(results):
    generated_data = [res[0] for res in results]
    latency_list = [res[2]-res[1] for res in results]
    throughput = len(results)/(results[-1][2] - results[0][1])

    generation_results = []
    for output, features, latency in zip(generated_data, result_data, latency_list):
        description = output['choices'][0]['text']
        score_info = calculate_total_score(output['choices'][0]['text'], features)

        sample_info = SampleGenerationResult(
            latency=latency,
            description=description,
            total_score=score_info['total_score'],
            detailed_score=copy.deepcopy(score_info['detailed_scores'])
        )
        generation_results.append(sample_info)

    full_generation_result = FullGenerationResult(
        throughput=throughput,
        latency_avg=sum([x.latency for x in generation_results])/len(generation_results),
        total_score_avg=sum([x.total_score for x in generation_results])/len(generation_results),
        outputs=generation_results
    )
    return full_generation_result

In [ ]:
##----------------------------------------------------------------------------------------------------------
results = await schedule_requests(input_data, pause_duration=4, model_name="cpatonn/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit")
full_generation_result = prepare_results_online(results)
print_compact_stats(full_generation_result)

#  СТАТИСТИКА ГЕНЕРАЦИИ
#    Throughput: 0.25 req/sec
#    Avg Latency: 2.661 sec
#    Samples: 100
#    Avg Score: 43.244

##----------------------------------------------------------------------------------------------------------
results = await schedule_requests(input_data, pause_duration=2, model_name="cpatonn/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit")
full_generation_result = prepare_results_online(results)
print_compact_stats(full_generation_result)

#  СТАТИСТИКА ГЕНЕРАЦИИ
#    Throughput: 0.50 req/sec
#    Avg Latency: 2.600 sec
#    Samples: 100
#    Avg Score: 43.030

##----------------------------------------------------------------------------------------------------------
results = await schedule_requests(input_data, pause_duration=1, model_name="cpatonn/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit")
full_generation_result = prepare_results_online(results)
print_compact_stats(full_generation_result)

#  СТАТИСТИКА ГЕНЕРАЦИИ
#    Throughput: 0.97 req/sec
#    Avg Latency: 3.325 sec
#    Samples: 100
#    Avg Score: 44.160

##----------------------------------------------------------------------------------------------------------
results = await schedule_requests(input_data, pause_duration=0.5, model_name="cpatonn/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit")
full_generation_result = prepare_results_online(results)
print_compact_stats(full_generation_result)

#  СТАТИСТИКА ГЕНЕРАЦИИ
#    Throughput: 1.88 req/sec
#    Avg Latency: 3.640 sec
#    Samples: 100
#    Avg Score: 44.418

##----------------------------------------------------------------------------------------------------------
results = await schedule_requests(input_data, pause_duration=0, model_name="cpatonn/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit")
full_generation_result = prepare_results_online(results)
print_compact_stats(full_generation_result)

#  СТАТИСТИКА ГЕНЕРАЦИИ
#    Throughput: 9.19 req/sec
#    Avg Latency: 9.741 sec
#    Samples: 100
#    Avg Score: 45.704